Fuentes: https://medium.com/nlplanet/fine-tuning-distilbert-on-senator-tweets-a6f2425ca50e

#### **Instalar Modulos**

conda install datasets=="2.20.0"

conda install transformers=="4.40.1"

conda install numpy=="1.26.4" # La última versión no funciona bien


In [1]:
#import torch
#print(torch.__version__)

In [2]:
#import numpy
#print(numpy.__version__)

In [3]:
#print(f"¿CUDA disponible?: {torch.cuda.is_available()}")
#if torch.cuda.is_available():
#    print(f"Dispositivo CUDA: {torch.cuda.get_device_name(0)}")
#    print(f"Versión de CUDA: {torch.version.cuda}")
#else:
#    print("CUDA no está disponible. PyTorch está usando la CPU.")


In [4]:
# Data processing
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
import copy

import time
import datetime

from sklearn.metrics import confusion_matrix, cohen_kappa_score

from datasets import Dataset,  DatasetDict

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

# Modeling
import torch
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizerFast, DataCollatorWithPadding, AutoModelForSequenceClassification, AdamW, get_scheduler

# Progress bar
from tqdm.auto import tqdm

# Add path to utils
import sys
sys.path.insert(0, '../tutoriales/')
from utils import plot_confusion_matrix, get_artifact_filename

from joblib import load, dump

# Verificamos que CUDA está funcional
torch.cuda.is_available()

d:\anaconda3\envs\ldi2_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

**Bajamos el modelo**

In [5]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

d:\anaconda3\envs\ldi2_cuda\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


**Armado de los Datasets**

In [6]:
# Paths
BASE_DIR = '../'
#PATH_TO_TRAIN = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train/train.csv")
PATH_TO_TRAIN = os.path.join(BASE_DIR, "work/cleaned/train_clean.csv")
PATH_TO_TEST = os.path.join(BASE_DIR, "work/cleaned/test_clean.csv")
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")

# Parametros y variables
SEED = 42
TEST_SIZE = 0.2

BATCH_SIZE = 16

MODEL_NAME = '01 DistilBert'

MODEL_VERSION = '3.0'

In [7]:
# Cargar los datos
df = pd.read_csv(PATH_TO_TRAIN)
df_test = pd.read_csv(PATH_TO_TEST)
df = df[df['Description'].notnull()]
df_test = df_test[df_test['Description'].notnull()]
df['labels'] = df["AdoptionSpeed"]
df_test['labels'] = df_test["AdoptionSpeed"]

# Dividir los datos usando sklearn
#train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, stratify=df.AdoptionSpeed)

try:
    study_lgb = optuna.create_study(direction='maximize',
                                storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
                                study_name="02 - DistilBert Optuna",
                               load_if_exists = True)
    
    # Verificar si el estudio tiene trials
    if len(study_lgb.trials) == 0:
        raise ValueError("No trials in study")
    
    lgb_test_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_lgb,'test')))
    
    train_df = df[~df.PetID.isin(lgb_test_dataset.PetID)].reset_index(drop=True)
    test_df = df[df.PetID.isin(lgb_test_dataset.PetID)].reset_index(drop=True)
    
except Exception as e:
    print(f"Error loading previous study: {e}. Using standard train_test_split.")
 #   train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, stratify=df.AdoptionSpeed)
    train_df = df 
    test_df = df_test

# Convertir a Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Combinar en un DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'val': test_dataset
})

# Codificar la columna de etiquetas como clases
dataset = dataset.class_encode_column('labels')

# Hacer una lista de columnas para remover antes de la tokenización
cols_to_remove = [col for col in dataset["train"].column_names if col != 'labels']
print(cols_to_remove)

[I 2026-04-28 10:13:12,576] Using an existing study with name '02 - DistilBert Optuna' instead of creating a new one.


Error loading previous study: No trials in study. Using standard train_test_split.


Casting to class labels: 100%|██████████| 2996/2996 [00:00<00:00, 285230.95 examples/s]

['Type', 'Name', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'Quantity', 'Fee', 'State', 'RescuerID', 'VideoAmt', 'Description', 'PetID', 'PhotoAmt', 'AdoptionSpeed', 'has_name', 'has_description', '__index_level_0__']


In [8]:
# Tokenize and encode the dataset
def tokenize(batch):
    from transformers import DistilBertTokenizerFast
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    tokenized_batch = tokenizer(batch["Description"], padding=True, truncation=True, max_length=512)
    return tokenized_batch

dataset_enc = dataset.map(tokenize, batched=True, remove_columns=cols_to_remove, num_proc=4)

# Set dataset format for PyTorch
dataset_enc.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# Check the output
print(dataset_enc["train"].column_names)
     


Map (num_proc=4): 100%|██████████| 2996/2996 [00:05<00:00, 539.57 examples/s]

['labels', 'input_ids', 'attention_mask']


In [9]:
# Instantiate a data collator with dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create data loaders for to reshape data for PyTorch model
train_dataloader = DataLoader(
    dataset_enc["train"], shuffle=True, batch_size=BATCH_SIZE, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    dataset_enc["val"], batch_size=BATCH_SIZE, collate_fn=data_collator
)

In [10]:
test_sample_ids =[i for i in test_df.PetID] 

In [11]:
# Dynamically set number of class labels based on dataset
num_labels = dataset["train"].features['labels'].num_classes
print(f"Number of labels: {num_labels}")

# Load model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", 
                                                           num_labels=num_labels)

d:\anaconda3\envs\ldi2_cuda\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Number of labels: 5


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:

# Set the device automatically (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Move model to device
model.to(device)

cuda


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [13]:
def train_val(model, dataloaders, datasets, device, trial=None):
    
    since = time.time()

    # Si trial es None, usar valores por defecto. Si no, usar Optuna para sugerir hyperparámetros
    if trial is None:
        lr = 2.5e-5
        weight_decay = 0.01
        num_epochs = 3
        print(f"Entrenamiento sin Optuna - LR: {lr}, Weight Decay: {weight_decay}, Epochs: {num_epochs}")
    else:
        # Usamos log=True para el LR porque varía en órdenes de magnitud
        lr = trial.suggest_float("lr", 1e-5, 5e-5, log=True)
        weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
        num_epochs = trial.suggest_int("num_epochs", 2, 5)
        print(f"Optuna Trial - LR: {lr}, Weight Decay: {weight_decay}, Epochs: {num_epochs}")

    # El batch_size se define usualmente fuera de esta función al crear los dataloaders, 
    # pero puedes obtenerlo para calcular los pasos de entrenamiento:
    train_dataloader = dataloaders['train']
    num_training_steps = num_epochs * len(train_dataloader)
    
    # 2. Configuración del Optimizador con Weight Decay
    optimizer = AdamW(
        model.parameters(), 
        lr=lr, 
        weight_decay=weight_decay
    )

    # 3. Scheduler con Warmup dinámico (10% de los pasos totales)
    num_warmup_steps = int(0.1 * num_training_steps)
    
    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
    )

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    best_kappa =  -999

    train_losses = []
    val_losses = []

    try:
        previous_best = study.best_value
    except:
        previous_best = -999


    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)
        
        kappa_labels_true = []
        kappa_labels_predicted = []
        output_scores = []

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for batch in tqdm(dataloaders[phase]):
                batch = batch.to(device)
                #inputs = inputs.to(device)
                labels = batch.labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward
                # Track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(**batch)
                    loss = outputs.loss

                    preds = torch.nn.functional.softmax(outputs.logits, dim=-1)
                    preds_labels = torch.argmax(preds, dim=-1)


                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                        lr_scheduler.step()     ################ ESTO LO AGREGÓ EL CHAT
                    elif phase == 'val':
                        kappa_labels_true.extend(labels.cpu().numpy().tolist())
                        kappa_labels_predicted.extend(preds_labels.cpu().numpy().tolist())
                        outputs_np = preds.cpu().numpy()
                        output_scores.extend([outputs_np[i,:] for i in range(outputs_np.shape[0])])

                # Statistics
                running_loss += loss.item() * labels.size(0)
                running_corrects += torch.sum(preds_labels == labels.data)
                
                # Liberar memoria GPU                                ################ ESTO LO AGREGÓ EL CHAT
                del batch, labels, outputs, preds, preds_labels
                torch.cuda.empty_cache()
                
                #END OF BATCH

            epoch_loss = running_loss / len(datasets[phase])
            epoch_acc = running_corrects.double() / len(datasets[phase])
            
            if phase == 'train':
                train_losses.append(epoch_loss)
                kappa_score = np.nan
            else:
                val_losses.append(epoch_loss)
                kappa_score = cohen_kappa_score(kappa_labels_true,
                                  kappa_labels_predicted,
                                  weights = 'quadratic')
                    


            print(f'{phase.title()} Loss: {epoch_loss:.4f} Acc: {epoch_acc*100:.2f}% Kappa: {kappa_score:.3f}')

            # If this is the best Epoch so far -> Deep copy the model
            if phase == 'val' and kappa_score > best_kappa:
                best_acc = epoch_acc
                best_kappa = kappa_score
                best_model_wts = copy.deepcopy(model.state_dict())


                #Best Epoch within a trial and better than previous trials
                if trial is not None and best_kappa > previous_best:

                    #Save test dataset with predictions
                    predicted_filename = os.path.join(PATH_TO_TEMP_FILES,f'test_{trial.study.study_name}_{trial.number}.joblib')
                    predicted_df = pd.DataFrame({'PetID':test_sample_ids,
                                'pred':output_scores}).merge(test_df, on='PetID')
                    dump(predicted_df, predicted_filename)

                    #Generate and save CM 
                    cm_filename = os.path.join(PATH_TO_TEMP_FILES,f'cm_{trial.study.study_name}_{trial.number}.jpg')
                    plot_confusion_matrix(kappa_labels_true,kappa_labels_predicted).write_image(cm_filename)

            #END OF PHASE

        #END OF EPOCH

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(
        time_elapsed // 60, time_elapsed % 60))
    print('Best val Acc: {:.2f}%'.format(best_acc * 100))

    # Load best model weights
    model.load_state_dict(best_model_wts)

    # Save in optuna trial the best test dataset, cm and model weights
    if trial is not None and best_kappa > previous_best:
        upload_artifact(trial, predicted_filename, artifact_store)   

        upload_artifact(trial, cm_filename, artifact_store)

        file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{trial.number}.pth'
        model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
        torch.save(model, model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()
        upload_artifact(trial, model_path, artifact_store)

    return model, best_kappa

In [14]:

# Dynamically set number of class labels based on dataset
num_labels = dataset["train"].features['labels'].num_classes
print(f"Number of labels: {num_labels}")


Number of labels: 5


In [ ]:

best_model,_ = train_val(model,
                       dataloaders={'train': train_dataloader, 
                                    'val': eval_dataloader}, 
                       datasets=dataset_enc, 
                       device=device, 
                       trial=None)


Entrenamiento sin Optuna - LR: 2.5e-05, Weight Decay: 0.01, Epochs: 3


d:\anaconda3\envs\ldi2_cuda\Lib\site-packages\transformers\optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 0/2
----------


100%|██████████| 749/749 [28:08<00:00,  2.25s/it]


Train Loss: 1.4600 Acc: 30.82% Kappa: nan


...

Val Loss: 1.3848 Acc: 37.25% Kappa: 0.244
Epoch 2/2
----------


100%|██████████| 749/749 [32:35<00:00,  2.61s/it]


Train Loss: 1.2096 Acc: 48.70% Kappa: nan


100%|██████████| 188/188 [00:34<00:00,  5.47it/s]

Val Loss: 1.4160 Acc: 37.85% Kappa: 0.234
Training complete in 95m 60s
Best val Acc: 37.25%


In [17]:
# Guardo el modelo
run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = f'{MODEL_NAME}_{MODEL_VERSION}_{run_id}.pth'
model_path = os.path.join(PATH_TO_TEMP_FILES, file_name)
torch.save(best_model, model_path) # Podemos guardar solo los pesos si queremos: best_model.state_dict()
print(f'Modelo guardado en {model_path}')

Modelo guardado en ../work/optuna_temp_artifacts\01 DistilBert_3.0_20260428_171644.pth


In [18]:
artifact_store = FileSystemArtifactStore(base_path=PATH_TO_OPTUNA_ARTIFACTS)


def optuna_train(trial):

    #epochs = trial.suggest_int('epochs', 1, 2)

    #lr = trial.suggest_float('lr', 0.00001, 0.0001, log=True)

    _,best_score = train_val(model, 
                       dataloaders={'train': train_dataloader, 
                                    'val': eval_dataloader}, 
                       datasets=dataset_enc, 
                       device=device, 
    #                   num_epochs=epochs,
     #                  lr=lr,
                       trial=trial)


    return(best_score)

In [19]:
study = optuna.create_study(direction='maximize',
                            storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
                            study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                            load_if_exists = True)
study.optimize(optuna_train, n_trials=5)



[I 2026-04-28 17:16:52,486] A new study created in RDB with name: 01 DistilBert_3.0
d:\anaconda3\envs\ldi2_cuda\Lib\site-packages\transformers\optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Optuna Trial - LR: 1.2632436159979682e-05, Weight Decay: 0.05038415833264941, Epochs: 3
Epoch 0/2
----------


100%|██████████| 749/749 [26:20<00:00,  2.11s/it]


Train Loss: 1.2155 Acc: 47.67% Kappa: nan


100%|██████████| 188/188 [00:34<00:00,  5.48it/s]


...

Val Loss: 3.2380 Acc: 35.98% Kappa: 0.239
Epoch 3/3
----------


100%|██████████| 749/749 [29:34<00:00,  2.37s/it]


Train Loss: 0.1173 Acc: 95.55% Kappa: nan


100%|██████████| 188/188 [00:35<00:00,  5.30it/s]
[I 2026-04-29 04:04:04,015] Trial 4 finished with value: 0.2394241452264717 and parameters: {'lr': 1.4154008570030158e-05, 'weight_decay': 0.055910564789734274, 'num_epochs': 4}. Best is trial 0 with value: 0.25451128228853215.


Val Loss: 3.2890 Acc: 36.35% Kappa: 0.234
Training complete in 123m 35s
Best val Acc: 35.98%
